# 06 - Cox Proportional Hazards Survival Analysis

Complement binary classification with Cox PH survival models.

**Analysis**:
1. Cox PH on GSE96058: pathway-only, clinical-only, combined features
2. 5-fold CV C-index for each feature set
3. Cox PH on TCGA (pathway-only, full-data C-index)
4. Kaplan-Meier curves with log-rank test

**Expected results**:
- GSE96058 pathway-only: C-index = 0.626 +/- 0.030
- GSE96058 clinical-only: C-index = 0.822 +/- 0.030
- GSE96058 combined: C-index = 0.827 +/- 0.031
- TCGA pathway-only: C-index = 0.590
- KM log-rank p < 5.7e-53

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_tcga_feature_matrix, load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features, build_feature_matrix
from src.survival import fit_cox_model, cox_cv, generate_km_curves

## 1. Prepare GSE96058 Data

In [ ]:
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

common = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
gse_clin = gse_clin[gse_clin['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common)].sort_values('sample_id').reset_index(drop=True)

gse_exp_norm = zscore_normalize(gse_exp)
pathway_features = compute_pathway_scores(gse_exp_norm)
pathway_features = add_ratio_features(pathway_features)
clinical_features = encode_clinical_features(gse_clin)
combined_features = build_feature_matrix(pathway_features, clinical_features)

time = gse_clin['time_to_event'].values
event = gse_clin['event_status'].values

print(f"Samples: {len(time)}")
print(f"Events: {event.sum()} ({event.mean():.1%})")

## 2. GSE96058 Cox PH Cross-Validation

In [ ]:
print("Cox PH 5-fold CV on GSE96058:\n")

feature_sets = {
    'Pathway Only': pathway_features,
    'Clinical Only': clinical_features,
    'Combined': combined_features,
}

cox_results = []
for name, X in feature_sets.items():
    mean_ci, std_ci, fold_cis = cox_cv(X, time, event)
    print(f"  {name:20s}  C-index: {mean_ci:.3f} +/- {std_ci:.3f}")
    cox_results.append({'Feature Set': name, 'C_index': mean_ci, 'C_index_SD': std_ci})

cox_df = pd.DataFrame(cox_results)
print("\n", cox_df.to_string(index=False))

## 3. TCGA Cox PH (Full Data)

In [ ]:
tcga = load_tcga_feature_matrix('../data/processed/02_tcga_feature_matrix.csv')
feature_cols = [c for c in tcga.columns if c.startswith('Pathway_') or c.startswith('Ratio_')]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
tcga_X = pd.DataFrame(
    scaler.fit_transform(tcga[feature_cols]),
    columns=feature_cols
)

tcga_cox = fit_cox_model(tcga_X, tcga['time_to_event'], tcga['event_status'])
tcga_cindex = tcga_cox.concordance_index_
print(f"TCGA pathway-only C-index (full data): {tcga_cindex:.3f}")

## 4. Kaplan-Meier Curves

In [ ]:
# Fit full Cox model on combined features for KM stratification
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(combined_features),
    columns=combined_features.columns
)
full_cox = fit_cox_model(X_scaled, time, event)

fig, p_value = generate_km_curves(
    X_scaled, time, event, full_cox,
    save_path='../figures/fig_kaplan_meier.png'
)
print(f"\nLog-rank p-value: {p_value:.2e}")
plt.show()

## 5. Summary

In [ ]:
print("=" * 60)
print("COX SURVIVAL ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nGSE96058 (5-fold CV):")
for _, row in cox_df.iterrows():
    print(f"  {row['Feature Set']:20s}  C-index: {row['C_index']:.3f} +/- {row['C_index_SD']:.3f}")
print(f"\nTCGA (full data):")
print(f"  Pathway Only           C-index: {tcga_cindex:.3f}")
print(f"\nKM log-rank p-value: {p_value:.2e}")